# Merge the LoRA Adapter into the Base Model

The fine-tune produced a **LoRA adapter** (~60 MB) that sits on top of the frozen Qwen2.5-3B base. For *serving*, it's cleaner to fold the adapter permanently into the base weights (`W_new = W + B·A`) so the result is a single standalone model that loads with plain `transformers` — no PEFT needed, and it's the form you quantize for CPU deployment.

This notebook loads the base + adapter, calls `merge_and_unload()`, and publishes the merged model to the Hugging Face Hub. Runs on Colab (a GPU runtime is fine; the merge itself is light).


## 1. Setup

Install the minimal stack (no `bitsandbytes`/`trl` — we merge in fp16, not 4-bit).

**Note on the `torchao` uninstall:** current `peft` has a version guard that requires `torchao > 0.16`, but Colab ships an older `torchao 0.10`, which makes `PeftModel` raise on import. We don't use `torchao` for an fp16 merge, so removing it lets the guard pass quietly.


In [ ]:
!pip install -q -U transformers peft huggingface_hub

In [ ]:
!pip uninstall -y torchao   # avoids peft's torchao>0.16 version guard; unused for fp16 merge

> ⚠️ **Restart the runtime** (Runtime → Restart session) after the setup cells before continuing — the in-place package changes need a fresh kernel.


## 2. Merge the adapter into the base

Load the base in **fp16** (merging needs full-precision weights to fold the adapter into — the standard QLoRA finish step), attach the trained adapter, then `merge_and_unload()` collapses `B·A` into `W`. The result is a normal Qwen model.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE    = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER = "tkatz123/qwen2.5-3b-job-extraction"

tok   = AutoTokenizer.from_pretrained(BASE)
base  = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER)      # base + trained adapter
merged = model.merge_and_unload()                     # fold ΔW = B·A into W -> standalone model

merged.save_pretrained("merged")
tok.save_pretrained("merged")
print("merged ✅  — standalone model, no PEFT needed")

## 3. Publish the merged model to the Hugging Face Hub

Pushes the ~6 GB fp16 model to a new repo. It then loads anywhere with `AutoModelForCausalLM.from_pretrained("tkatz123/qwen2.5-3b-job-extraction-merged")` — including the inference service.


In [ ]:
from huggingface_hub import notebook_login, create_repo, HfApi, whoami
notebook_login()      # paste a WRITE token into the widget

REPO = f'{whoami()["name"]}/qwen2.5-3b-job-extraction-merged'
create_repo(REPO, repo_type="model", exist_ok=True)
HfApi().upload_folder(folder_path="merged", repo_id=REPO, repo_type="model")
print("pushed ->", f"https://huggingface.co/{REPO}")